In [1]:
import torch

if torch.cuda.is_available():
  print("Success! GPU is active and ready to process video.")
else:
  print("Uh oh. The GPU is not active. Check Step 2 again.")

Success! GPU is active and ready to process video.


In [2]:
import os
from google.colab import userdata

# 1. Inject your Kaggle credentials directly into the environment
os.environ['KAGGLE_USERNAME'] = 'syedabdulnafay'
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

# 2. Install the Kaggle library quietly (-q)
!pip install -q kaggle

# 3. Download the SCVD dataset directly from Kaggle's servers
print('Downloading SCVD dataset...')
!kaggle datasets download -d toluwaniaremu/smartcity-cctv-violence-detection-dataset-scvd

# 4. Unzip the downloaded file into a new folder called 'scvd_dataset'
print('Unzipping dataset... this might take a minute...')
!unzip -q smartcity-cctv-violence-detection-dataset-scvd.zip -d scvd_dataset

print('Success! Data acquired.')

Dataset URL: https://www.kaggle.com/datasets/toluwaniaremu/smartcity-cctv-violence-detection-dataset-scvd
License(s): CC-BY-NC-SA-4.0
100% 1.03G/1.03G [00:55<00:00, 19.9MB/s]

Unzipping dataset... this might take a minute...
Success! Data acquired.


In [ ]:
import os
import time
import logging
import random
import hashlib
import cv2
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

# --- PROJECT MODULES ---------------------------------------------------------
# The data pipeline now lives in the repo, not in this notebook, so training,
# evaluation and the live system share ONE implementation. Upload these files
# from the repo root to the Colab working directory (Files pane), or
# `git clone` the repo and `%cd` into it.
REQUIRED_MODULES = [
    'thresholds.py', 'clip_sampler.py', 'incident.py', 'frame_prep.py',
    'prepared_frames.py', 'frame_cache.py', 'clip_eval.py',
    'inference_core.py', 'cached_dataset.py', 'preprocessing.py',
]
_missing = [m for m in REQUIRED_MODULES if not os.path.exists(m)]
if _missing:
    raise ImportError(
        f"Missing project files in {os.getcwd()}: {_missing}. Upload them from the "
        "repo (same versions as the backend) before running this cell."
    )

from frame_prep import prepare_for_model
from frame_cache import FrameCache
from cached_dataset import CacheItem, CachedClipDataset, iter_eval_videos, seed_worker
import clip_eval
import preprocessing
from preprocessing import PREPROC_V1, PREPROC_V2, PreprocSpec

# --- MODEL INPUT CONTRACT ------------------------------------------------------
# How frames are cropped and how pixels are scaled. It is saved next to the
# weights (sidecar .json) and the backend reads it from there, so the live
# system always feeds a model the input it was trained on.
#   PREPROC_V1 = PreprocSpec('frame', 'none'): every frame cropped on its own,
#       /255 only -- the original pipeline (the baseline run).
#   PREPROC_V2 = PreprocSpec('clip', 'kinetics'): ONE crop box per 16-frame
#       clip (from the motion of all its frames), then Kinetics-400 mean/std.
#       Why: (1) per-frame crops jump between motion boxes and the centre
#       fallback inside a clip, which looks like camera zoom/pan -- and
#       violent clips jump more, a shortcut the model can learn (measured on
#       test videos: a calm clip had as much frame-to-frame change as a
#       violent one purely from crop jumps). Kinetics clips use one crop per
#       clip. (2) the frozen layers were pretrained on mean/std-normalized
#       input; /255-only input is out of their distribution.
# The fields are independent, so each change can be ablated on its own, e.g.
# PreprocSpec('clip', 'none') or PreprocSpec('frame', 'kinetics').
PREPROCESSING = PREPROC_V2

# Keep BatchNorm running statistics of the FROZEN layers (stem..layer3) fixed.
# model.train() otherwise keeps updating them from batches of 8 clips even
# though their weights are frozen. With Kinetics normalization the
# pretrained statistics are exactly right, so drifting them only adds noise.
# (PREPROC_V1 + False reproduces the baseline run.)
FREEZE_FROZEN_BN = True

# Show frame-cache build progress in the cell output.
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s', force=True)

# SANITY CHECK -- fail loudly on a stale frame_prep.py. prepare_for_model
# returns 3 values (rgb, meta, gray) with the person box in meta["person_box"];
# an old 4-value copy would otherwise die deep inside the cache build.
_dummy_out = prepare_for_model(np.zeros((240, 320, 3), np.uint8), None, run_person_detector=False)
assert len(_dummy_out) == 3, (
    f"prepare_for_model returned {len(_dummy_out)} values, expected 3 "
    "(rgb, meta, gray). Re-upload the updated frame_prep.py to Colab."
)
assert "person_box" in _dummy_out[1], "frame_prep.py is stale: meta has no 'person_box'"
del _dummy_out

# REPRODUCIBILITY. Seed python/numpy/torch once, and give every DataLoader a
# seeded torch.Generator (shuffle order) plus seed_worker (per-worker `random`
# / numpy state, used for the random training window and the flip).
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# With the frame cache a training item is a memory-mapped read of 16 small
# frames, no decoding, so 2 workers are plenty to keep the GPU busy.
NUM_WORKERS = 2

def make_loader(ds, batch_size, shuffle):
    g = torch.Generator()
    g.manual_seed(SEED)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=True,
                      worker_init_fn=seed_worker, generator=g)

# --- FRAME CACHE ---------------------------------------------------------------
# Previously every __getitem__ decoded up to 340 raw frames and ran frame_prep
# on them to use 16 -- every access, every epoch. That CPU work (2 Colab
# cores) was the bottleneck, not the GPU.
# Now each video is prepared ONCE by prepared_frames.PreparedFrameReader (the
# same front-end /analyze/video uses: 15 fps timeline resampling ->
# prepare_for_model with HOG off) and stored. A window cut from the cache is
# bit-identical to preparing it on the fly, so this changes speed, not results.
# What is cached follows PREPROCESSING.crop: final 112x112 crops ('frame'),
# or working frames (short side 224) + motion boxes ('clip'; each training
# window is clip-cropped on read, exactly like the live camera).
#   - 'auto' (default): 'npy' for 'frame' mode, 'jpg' for 'clip' mode.
#   - 'npy': exact; ~6 MB per 10 s clip in 'frame' mode, ~40 MB in 'clip'
#     mode -> keep it on the LOCAL disk. Rebuilt each new Colab session.
#   - 'jpg': 10-25x smaller, fits on Drive and survives restarts, slightly
#     LOSSY (the live camera's own frames are JPEG-compressed too).
# Entries are keyed by video CONTENT + a fingerprint of the frame code, so a
# changed frame_prep.py can never serve stale frames.
CACHE_DIR = '/content/frame_cache'
CACHE_FORMAT = 'auto'
CACHE_WORKERS = 2  # processes used to build the cache (Colab has 2 cores; 0 = in-process)
# The old MAX_RAW_FRAMES=340 cap existed because long Violent clips were being
# truncated before their violent part. The cache stores whole clips, so there
# is no cap anymore.
#
# SMART CROP: always on now (prepare_for_model is the only front-end). The old
# bare-resize A/B path is gone -- it never matched what live cameras feed the
# model. The diagnostic cells below still document why HOG stays OFF.

# --- VALIDATION HOLD-OUT (leak-free) ---
# There is no validation set in the manifest (only train/test). We carve ~15%
# of the TRAIN rows out as validation and never train on them. The test split
# stays untouched for the final report.
#
# Near-duplicate clips (the same source video cut into several segments)
# must land on the SAME side of the split, otherwise validation scores are
# inflated by leakage. So the split is done at the cluster level:
#   1. If the manifest has a column whose name contains 'cluster' or
#      'group', hash that value (md5) to decide train vs val.
#   2. Otherwise fall back to hashing the clip path (per-clip, not
#      per-cluster -- still deterministic, but weaker against near-duplicates).
# Hashing (rather than random sampling) means the split never changes
# between runs or Colab restarts.

def find_cluster_column(df):
    """Return the first manifest column whose name contains 'cluster' or 'group', else None."""
    for col in df.columns:
        low = str(col).lower()
        if 'cluster' in low or 'group' in low:
            return col
    return None

def _md5_bucket(key, buckets=10000):
    return int(hashlib.md5(str(key).encode('utf-8')).hexdigest(), 16) % buckets

def make_train_val_split(manifest_path, val_frac=0.15):
    df = pd.read_csv(manifest_path)
    df_train = df[df['split'] == 'train'].copy()
    col = find_cluster_column(df_train)
    if col is not None:
        keys = df_train[col].where(df_train[col].notna(), df_train['path']).astype(str)
        approach = f"cluster column '{col}' ({keys.nunique()} distinct groups)"
    else:
        keys = df_train['path'].astype(str)
        approach = "md5 hash of clip path (no cluster/group column found in manifest)"
    is_val = keys.map(_md5_bucket) < int(val_frac * 10000)
    print(f"Validation split approach: {approach}")
    return df_train[~is_val].copy(), df_train[is_val].copy(), approach

# --- LABELS: 2 classes, deliberately ---
# The 3-class run showed the dominant confusion was Violent<->Weaponized (32%
# of real Violent clips called Weaponized, 14% the other way) -- not
# danger-vs-safe. Motion alone doesn't reliably carry "was there also a small
# handheld object"; that's an appearance question. So this model's only job
# is Danger-vs-Normal; weapon-specific urgency is meant to come from a
# SEPARATE weapon-presence signal applied only when this model already says
# Dangerous.
#
# NOTE: 'Weaponized' is spelled correctly here to match the real folder name.
# Earlier runs used a typo ('Weaponzied') that silently matched no folder, so
# every run before this notebook trained "Violent" on Violence clips only.
CLASS_MAPPING = {'Normal': 0, 'Violence': 1, 'Weaponized': 1}

def manifest_items(df, keys):
    """Manifest rows -> CacheItems, skipping videos the cache could not read."""
    return [CacheItem(key=keys[p], label=CLASS_MAPPING[c], path=p, cls=c)
            for p, c in zip(df['path'], df['class']) if p in keys]

# --- TRAINING SAMPLES ---
# Each item = one uniformly random 16-frame window over the WHOLE clip (a new
# one every epoch) + a horizontal flip with p=0.5 (same flip for all frames).
# Full-range jitter is kept on purpose: bounded/centre-anchored jitter was
# tested head-to-head and cut recall from 86% to 73% for +2 points precision,
# because the violent action in these clips is not reliably centred.
WINDOW_LEN = 16

In [ ]:
# Path to the manifest you generated with rebuild_clean_split.py
manifest_file = 'clean_split_manifest.csv'

# Hold ~15% of the TRAIN rows out as a validation set (cluster-level if the
# manifest has a cluster/group column, else a path-hash fallback -- the
# approach used is printed below). The test split is left alone.
train_df, val_df, split_approach = make_train_val_split(manifest_file, val_frac=0.15)
_manifest = pd.read_csv(manifest_file)
test_df = _manifest[_manifest['split'] == 'test'].copy()

# Build (or reuse) the frame cache for every clip once, up front.
cache = FrameCache(CACHE_DIR, fmt=CACHE_FORMAT, crop_mode=PREPROCESSING.crop)
print(f"Preprocessing: {PREPROCESSING.name} (crop={PREPROCESSING.crop}, normalize={PREPROCESSING.normalize})")
print(f"Frame cache: {CACHE_DIR} (format={cache.fmt}, fingerprint={cache.fingerprint[:12]})")
all_paths = pd.concat([train_df, val_df, test_df])['path'].tolist()
cache_report = cache.build_many(all_paths, workers=CACHE_WORKERS, log_every=250)
print(cache_report)
for p, err in list(cache_report.failed.items())[:10]:
    print(f"  unreadable, skipped: {p} ({err})")

train_items = manifest_items(train_df, cache_report.keys)
val_items = manifest_items(val_df, cache_report.keys)
test_items = manifest_items(test_df, cache_report.keys)

# Training set: random window + flip. Validation/test are not Datasets any
# more -- they are scored clip-by-clip on the live schedule (see the training
# cell), so they are just lists of cached videos.
my_training_data = CachedClipDataset(cache, train_items, windows_per_video=1, hflip=True,
                                     preprocessing=PREPROCESSING)
train_loader = make_loader(my_training_data, batch_size=8, shuffle=True)

print(f"Train clips: {len(train_items)} | Validation clips: {len(val_items)} | Test clips: {len(test_items)}")
for name, items in [('train', train_items), ('val', val_items), ('test', test_items)]:
    print(f"  {name}: " + ", ".join(f"{c}={sum(it.cls == c for it in items)}" for c in CLASS_MAPPING))

t0 = time.time()
first_video_tensor, first_video_label = my_training_data[0]
dt = time.time() - t0

print('---')
print(f'Label: {first_video_label} (0=Normal, 1=Violent)')
print(f'Tensor Shape: {first_video_tensor.shape}')
print(f'__getitem__ took {dt*1000:.1f} ms (cached; it used to take seconds)')
print('If the shape says [3, 16, 112, 112], your data pipeline is absolutely perfect!')

In [ ]:
# --- DIAGNOSTIC: clip lengths and how much room the random window has ---
# Read straight from the cache (no decoding): length on the 15 fps timeline,
# how many windows live mode would score per clip (the validation/test
# schedule), and how many distinct start points a random training window can
# take ("jitter room"). Large jitter room on Violent clips = a training window
# can land entirely on calm lead-in/aftermath of a clip labelled Violent.
rows = []
for it in train_items:
    v = cache.load(it.key)
    rows.append({'class': it.cls, 'frames_15fps': len(v), 'seconds': len(v) / v.fps,
                 'live_windows': len(v.schedule()), 'jitter_room': len(v.ready_windows())})
_diag = pd.DataFrame(rows)
print(_diag.groupby('class')[['frames_15fps', 'seconds', 'live_windows', 'jitter_room']]
           .describe(percentiles=[0.5]).round(1).to_string())
print()
print("Read this as: 'live_windows' is how many windows the deployment-style")
print("evaluation scores for a clip (one every 8 frames). 'jitter_room' is how")
print("many different 16-frame windows a training sample can be drawn from.")

In [ ]:
# --- DIAGNOSTIC: does the live-pipeline smart-crop actually matter for
# THIS dataset? ---
# Production (frame_prep.prepare_for_model, used in camera.py/main.py)
# detects the person/motion in a raw frame and crops+zooms to them BEFORE
# resizing to 112x112, so subjects stay a meaningful fraction of the model
# input even from a wide 1080p hallway shot. Training here just resizes the
# raw SCVD frame directly -- no cropping. If SCVD clips are already
# close/medium shots, this mismatch barely matters. If they're wide shots,
# it's a real train/inference distribution gap, same category of bug as
# the normalization mismatch we found earlier.
#
# This cell measures it directly instead of guessing: samples a few short
# bursts of frames from a subset of clips per class, runs them through the
# ACTUAL production cropping function, and reports what fraction of the
# frame the subject ends up occupying, and how often frame_prep would even
# call the scene "suitable_for_model" as-is.
#
# REQUIRES frame_prep.py to be present in this Colab session (it's not
# part of the training notebook by default -- upload it alongside your
# dataset, e.g. via the Files pane or `files.upload()`, before running this).
import random as _random
import numpy as _np

try:
    from frame_prep import prepare_for_model
except ImportError:
    raise ImportError(
        "frame_prep.py not found in this Colab session. Upload it (same file "
        "used by camera.py in production) to the working directory, then "
        "re-run this cell."
    )

SAMPLE_CLIPS_PER_CLASS = 15   # keep this small -- HOG is the slow part
BURST_LEN = 5                 # consecutive raw frames per sample point
ANCHOR_FRACTIONS = [0.25, 0.5, 0.75]  # sample near start/middle/end of each clip

def sample_scene_stats(video_paths, n_clips):
    paths = video_paths if len(video_paths) <= n_clips else _random.sample(video_paths, n_clips)
    methods = []
    subject_fracs = []
    suitable_flags = []
    for path in paths:
        cap = cv2.VideoCapture(path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames < BURST_LEN + 1:
            cap.release()
            continue
        for frac in ANCHOR_FRACTIONS:
            start_idx = max(0, min(total_frames - BURST_LEN, int(total_frames * frac)))
            cap.set(cv2.CAP_PROP_POS_FRAMES, start_idx)
            prev_gray = None
            for _ in range(BURST_LEN):
                ok, frame = cap.read()
                if not ok:
                    break
                # HOG is deliberately LEFT ON here (default
                # run_person_detector=True): this audit is what measures
                # whether person_crop ever fires. Training cells turn it off.
                rgb112, meta, prev_gray = prepare_for_model(
                    frame, prev_gray
                )
            else:
                # only record the LAST frame of a completed burst -- it's the
                # one with valid prev_gray, so motion_crop gets a fair chance too
                methods.append(meta['method'])
                subject_fracs.append(meta['subject_area_frac'])
                suitable_flags.append(meta['suitable_for_model'])
        cap.release()
    return methods, subject_fracs, suitable_flags

print("Scanning a sample of raw SCVD_converted frames through the production "
      "cropping pipeline (frame_prep.prepare_for_model)...")
print("This can take a minute or two -- HOG detection is the slow part.\n")

from collections import Counter

for class_name, label_name in [('Normal', 'Normal'), ('Violence', 'Violence'), ('Weaponized', 'Weaponized')]:
    # Filter by the manifest's raw class NAME. (The old filter compared the
    # merged 0/1 label against 2, which no longer exists, so Weaponized was
    # never sampled.)
    paths = [p for p, c in zip(my_training_data.video_paths, my_training_data.classes)
             if c == class_name]
    if not paths:
        continue
    methods, fracs, suitable = sample_scene_stats(paths, SAMPLE_CLIPS_PER_CLASS)
    if not methods:
        print(f"{label_name}: no valid samples collected")
        continue
    method_counts = Counter(methods)
    pct_suitable = 100 * sum(suitable) / len(suitable)

    # IMPORTANT: frame_prep hardcodes subject_area_frac=0.0 whenever it falls
    # back to center_crop (no person or motion detected) -- that 0.0 is a
    # placeholder, not a measurement. Blending it into an aggregate min/
    # median/max makes "no detection happened" look like "detected a tiny
    # subject", which are very different findings. Split them apart:
    detected_fracs = _np.array([f for m, f in zip(methods, fracs) if m != 'center_crop'])
    n_no_detection = method_counts.get('center_crop', 0) + method_counts.get('full_frame', 0)
    pct_no_detection = 100 * n_no_detection / len(methods)

    print(f"--- {label_name} ({len(methods)} sampled frames from up to {SAMPLE_CLIPS_PER_CLASS} clips) ---")
    print(f"  Crop method breakdown: {dict(method_counts)}")
    print(f"  No detection at all (center_crop/full_frame fallback): {pct_no_detection:.1f}% of samples")
    if len(detected_fracs) > 0:
        print(f"  subject_area_frac WHEN SOMETHING WAS DETECTED (n={len(detected_fracs)}): "
              f"min={detected_fracs.min():.3f} median={_np.median(detected_fracs):.3f} max={detected_fracs.max():.3f}")
    else:
        print("  subject_area_frac WHEN SOMETHING WAS DETECTED: n/a (nothing was ever detected)")
    print(f"  suitable_for_model (per frame_prep's own threshold, all samples): {pct_suitable:.1f}%")
    print()

print("How to read this:")
print("- 'No detection at all' % is its own finding, separate from subject")
print("  size: it tells you how often frame_prep found NOTHING to crop to,")
print("  in which case it silently falls back to a plain center crop. High")
print("  numbers here (for either class) mean a large chunk of frames never")
print("  even reach the crop-quality question -- they're centered-and-hoped,")
print("  not smart-cropped.")
print("- 'subject_area_frac WHEN SOMETHING WAS DETECTED' is the honest read")
print("  on subject scale, restricted to frames where a real box existed.")
print("  Compare its median to MIN_SUBJECT_AREA_FRAC=0.05.")
print("- If person_crop count is 0 or near-0 across both classes: the HOG")
print("  pedestrian detector (trained on upright, street-level walking")
print("  people) likely doesn't fire reliably on this footage's camera")
print("  angle/poses. That is a PRODUCTION-relevant finding on its own,")
print("  independent of the training question -- if live cameras share a")
print("  similar mounting angle, person_crop may rarely fire there either,")
print("  meaning the live system is leaning almost entirely on the motion")
print("  fallback (and plain center_crop when even that fails) in practice.")
print("  Worth checking against real camera footage separately.")


In [7]:
import torch
import torch.nn as nn
import torchvision.models.video as video_models

print("Downloading the pre-trained R3D_18 model (This might take a few seconds)...")

# 1. LOAD THE PRE-TRAINED MODEL
# We pull the model and its pre-learned Kinetics-400 weights
weights = video_models.R3D_18_Weights.DEFAULT
model = video_models.r3d_18(weights=weights)

# 2. FREEZE THE KNOWLEDGE (Transfer Learning)
# We loop through all the existing "neurons" and tell PyTorch NOT to update them.
# This saves massive computing power and prevents the model from forgetting the basics.
for param in model.parameters():
  param.requires_grad = False

# 2.5 PARTIAL UNFREEZE (layer4) -- TOGGLE FOR ABLATION
# r3d_18 is stem -> layer1 -> layer2 -> layer3 -> layer4 -> avgpool -> fc.
# layer1-3 learn low-level stuff (edges, textures, generic motion primitives)
# that's the same whether the action is a handshake or a punch -- no reason
# to retrain those. layer4 is the deepest residual block group (it contains
# 2 residual blocks) and is where the highest-level spatiotemporal features
# live -- exactly the level where "violent contact" vs "generic fast human
# motion" needs to be distinguished.
#
# UNFREEZE_LAYER4 = False reproduces the original fc-only setup (on top of
# whatever dataset/jitter/weighting is currently configured elsewhere in
# the notebook). Run it both ways with everything else held constant to
# see how much of the precision/recall shift across your recent runs is
# actually attributable to this layer, versus the dataset swap + jitter.
UNFREEZE_LAYER4 = True  # confirmed better than False: see Run 3 vs Run 4 comparison

if UNFREEZE_LAYER4:
  for param in model.layer4.parameters():
    param.requires_grad = True
  print("layer4: UNFROZEN (trainable)")
else:
  print("layer4: frozen (fc-only training, like the original baseline)")

# 3. THE SURGERY (Replacing the Head)
# We find out how many connections are flowing into the final layer
num_features = model.fc.in_features

# We replace the old 400-class layer with a new 2-class layer (0: Normal, 1: Violent)
model.fc = nn.Linear(num_features, 2)

# We make sure the this brand new layer is unlocked and ready to learn
for param in model.fc.parameters():
  param.requires_grad = True

# Sanity check: confirm what's actually trainable before you spend GPU time
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")
print("Expect this to include layer4.* and fc.* only -- if it's just fc, the unfreeze above didn't take.")

# 4. MOVE TO THE SUPERCOMPUTER
# We tell PyTorch to grab our newly modified model and throw it onto the GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("\n--- Surgery Complete ---")
print(f"Model successfully moved to: {device}")
print("If the device above says 'cuda', your GPU is active and the brain is ready!")

Downloading: "https://download.pytorch.org/models/r3d_18-b3b3357e.pth" to /root/.cache/torch/hub/checkpoints/r3d_18-b3b3357e.pth


100%|██████████| 127M/127M [00:00<00:00, 186MB/s]


layer4: UNFROZEN (trainable)
Trainable params: 24,909,826 / 33,167,298 (75.1%)
Expect this to include layer4.* and fc.* only -- if it's just fc, the unfreeze above didn't take.

--- Surgery Complete ---
Model successfully moved to: cuda
If the device above says 'cuda', your GPU is active and the brain is ready!


In [ ]:
# DRIVE CHECKPOINT: a closed laptop lid / browser disconnect can kill the
# Colab session and wipe in-memory weights. Mount Drive up front so the best
# epoch is written to MyDrive/safespace_checkpoints/ every time it improves.
# Recover later with: ckpt = torch.load(path); model.load_state_dict(ckpt["state_dict"])
import os
CHECKPOINT_DIR = None
try:
  from google.colab import drive
  drive.mount("/content/drive")
  CHECKPOINT_DIR = "/content/drive/MyDrive/safespace_checkpoints"
  os.makedirs(CHECKPOINT_DIR, exist_ok=True)
  print(f"Best-epoch checkpoints will be saved to {CHECKPOINT_DIR}")
except Exception as e:
  print(f"WARNING: Google Drive not mounted ({e}) -- best weights will only live in memory.")

import copy
import math
import torch.optim as optim

print("Setting up the classroom...")

# The model carries its input contract, so evaluation (clip_eval) crops and
# normalizes validation clips exactly like training and the live system.
preprocessing.attach(model, PREPROCESSING)

# Modules whose weights are frozen (see the surgery cell). With
# FREEZE_FROZEN_BN their BatchNorm layers stay in eval mode during training.
frozen_modules = [model.stem, model.layer1, model.layer2, model.layer3]
if not UNFREEZE_LAYER4:
  frozen_modules.append(model.layer4)

def set_train_mode(model):
  model.train()
  if FREEZE_FROZEN_BN:
    for m in frozen_modules:
      m.eval()

# 1. BATCHING (The DataLoader)
# Video tensors are massive. If we try to feed all our videos to the GPU at once,
# it will instantly crash (Out of Memory error). We group them into tiny batches of 8.
batch_size = 8
# make_loader (defined in the dataset cell) = num_workers=2, pin_memory=True,
# seeded worker_init_fn and a seeded torch.Generator for the shuffle order.
train_loader = make_loader(my_training_data, batch_size=batch_size, shuffle=True)

# 2. THE GRADER & THE STUDENT
# CrossEntropyLoss is the standard mathematical formula for grading multiple-choice questions
#
# CLASS WEIGHTING. History (kept for the report):
#  - Weighting was first added against a recall collapse that turned out to be
#    mostly a truncation bug (MAX_RAW_FRAMES cut Violent clips short).
#  - Run 6 (unweighted, truncation fixed): recall collapsed to 55%, so the
#    imbalance was real and weighting stayed.
#  - Run 7: THE IMBALANCE FLIPPED once the Weaponized clips were (correctly)
#    loaded -- Normal ~1029 vs Dangerous ~1653 -- so Normal recall (false
#    alarms) became the weak metric. Softened (sqrt) weights now push in the
#    right direction: roughly [1.14, 0.90]. sqrt rather than full
#    inverse-frequency, because full-strength weighting just trades one error
#    for the other.
USE_CLASS_WEIGHTS = True

label_counts = torch.bincount(torch.tensor(my_training_data.labels))
num_classes = len(label_counts)
num_samples = len(my_training_data.labels)
raw_weights = num_samples / (num_classes * label_counts.float())
sqrt_weights = raw_weights.sqrt()
print(f"Class counts (Normal, Dangerous) in the TRAIN subset: {label_counts.tolist()}")
print(f"Raw inverse-frequency weights (unused unless noted): {raw_weights.tolist()}")
print(f"Softened (sqrt) weights (unused unless noted): {sqrt_weights.tolist()}")

if USE_CLASS_WEIGHTS:
    class_weights = sqrt_weights.to(device)
    print(f"USE_CLASS_WEIGHTS=True -> training with weights: {class_weights.tolist()}")
    criterion = nn.CrossEntropyLoss(weight=class_weights)
else:
    print("USE_CLASS_WEIGHTS=False -> plain unweighted CrossEntropyLoss")
    criterion = nn.CrossEntropyLoss()

# The Optimizer is the student. Two param groups since layer4 is unfrozen:
#   - model.fc: brand new, random-initialized layer -> normal LR (1e-3)
#   - model.layer4: pretrained Kinetics-400 weights -> a much smaller LR (1e-5)
#     so we nudge them toward violence-specific features instead of
#     overwriting what they already know.
param_groups = [{'params': model.fc.parameters(), 'lr': 0.001}]
if UNFREEZE_LAYER4:
    param_groups.append({'params': model.layer4.parameters(), 'lr': 0.00001})
optimizer = optim.Adam(param_groups)

# 3. THE LOOP
# We keep the BEST epoch (by validation), not the last one.
epochs = 8

# PER-EPOCH VALIDATION -- DEPLOYMENT-ALIGNED.
# The old check scored only the FIRST 16-frame window of each clip at 0.5.
# Live mode scores a window every 8 frames and alerts through IncidentTracker
# (spike >= 0.90 confirmed by the next window >= 0.67, or the 4-window average
# >= 0.67 twice in a row). clip_eval scores every validation clip exactly that
# way and reports:
#   - PR-AUC / ROC-AUC of per-clip scores (first / max / mean window, and the
#     peak rolling average),
#   - Dangerous recall at Normal recall >= 0.90 (i.e. at <= 10% false alarms),
#   - the live incident rule's own verdict (would_alert) as a confusion matrix.
# The best epoch is the one with the highest SELECTION_METRIC. Default
# 'pr_auc_max' is threshold-free on purpose: thresholds are recalibrated for
# every new model (video_calibration.py), so selecting on today's 0.67/0.90
# would mix up "better model" with "better fit to old thresholds".
# Options: clip_eval.SELECTION_METRICS.keys()
SELECTION_METRIC = clip_eval.DEFAULT_SELECTION_METRIC
EVAL_BATCH_SIZE = 32

def evaluate_split(model, items):
  """Score every live-schedule window of every clip, replay the incident rule."""
  return clip_eval.evaluate_model(model, iter_eval_videos(cache, items), device,
                                  batch_size=EVAL_BATCH_SIZE)

best_val_value = -math.inf
best_epoch = 0
best_state = None          # in-memory copy of the best weights (copy.deepcopy)
best_val_metrics = None    # full clip_eval summary of the best epoch
history = []

print("Starting the Training Loop! Grab a coffee, this will take a few minutes...")

for epoch in range(epochs):
  set_train_mode(model) # "Learning Mode" (frozen BatchNorm stays fixed if FREEZE_FROZEN_BN)
  running_loss = 0.0

  # We loop through our dataset, grabbing 8 videos (inputs) and 8 answers (labels) at a time
  for i, (inputs, labels) in enumerate(train_loader):

    # Move the data from the CPU's RAM onto the GPU's ultra-fast memory
    inputs = inputs.to(device)
    labels = labels.to(device)

    # Step A: Wipe the slate clean
    optimizer.zero_grad()

    # Step B: The Forward Pass (Make a guess)
    outputs = model(inputs)

    # Step C: Calculate the Error (How wrong was the guess?)
    loss = criterion(outputs, labels)

    # Step D: The Backward Pass (Learn from the mistake)
    loss.backward()

    # Apply the adjustments
    optimizer.step()

    running_loss += loss.item()

    # Print a progress update every 10 batches so we know it hasn't frozen
    if (i + 1) % 10 == 0:
      print(f"Epoch [{epoch+1}/{epochs}], Batch [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}")

  epoch_loss = running_loss / len(train_loader)
  print(f"--- End of Epoch {epoch + 1} | Average Loss: {epoch_loss:.4f} ---")

  # Validate the way the model is used live, and remember the best epoch.
  val_res = evaluate_split(model, val_items)
  value = clip_eval.selection_value(val_res.summary, SELECTION_METRIC)
  live = val_res.summary['live']
  history.append({'epoch': epoch + 1, 'train_loss': epoch_loss, SELECTION_METRIC: value,
                  'live_normal_recall': live['normal_recall'],
                  'live_dangerous_recall': live['dangerous_recall'],
                  'live_macro_f1': live['macro_f1']})
  print(clip_eval.format_summary(val_res.summary))
  if value > best_val_value:
    best_val_value = value
    best_epoch = epoch + 1
    best_val_metrics = val_res.summary
    best_state = copy.deepcopy(model.state_dict())
    print(f"    ** new best (epoch {best_epoch}, {SELECTION_METRIC}={value:.4f}) -- weights kept in memory **")
    if CHECKPOINT_DIR:
      _ckpt = os.path.join(CHECKPOINT_DIR, "safespace_v3_best_checkpoint.pth")
      torch.save({"state_dict": best_state, "epoch": best_epoch, "val_metrics": best_val_metrics,
                  "selection_metric": SELECTION_METRIC}, _ckpt)
      print(f"    ** also saved to Google Drive: {_ckpt} **")

print(f"\n Training Complete! Best epoch: {best_epoch} ({SELECTION_METRIC} {best_val_value:.4f}). "
      "The evaluation cell loads these weights, not the last epoch's.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# --- TEST-SET DISCIPLINE ---
# Every design decision (jitter, weighting, unfreezing, thresholds...) is made
# on VALIDATION. The test split is scored ONCE, at the very end, for the
# report. Comparing runs on test and keeping whichever looks best quietly
# leaks the test set into the design and inflates the final number.
# Leave this False while iterating: the cell then reports on validation.
RUN_FINAL_TEST = False

EVAL_SPLIT = 'test' if RUN_FINAL_TEST else 'val'
eval_items = test_items if RUN_FINAL_TEST else val_items
print(f"Evaluating on the {EVAL_SPLIT.upper()} split ({len(eval_items)} clips)"
      + ("" if RUN_FINAL_TEST else " -- set RUN_FINAL_TEST=True once, for the final report"))

# 0. Load the BEST epoch's weights (by validation), not whatever the last
# epoch left in the model.
model.load_state_dict(best_state)
print(f"Loaded best weights: epoch {best_epoch}, {SELECTION_METRIC} {best_val_value:.4f}")

# 1. Score every clip on the live schedule (a window every 8 frames over the
# WHOLE clip) and replay the live incident rule.
eval_res = evaluate_split(model, eval_items)
results_df = pd.DataFrame(eval_res.rows)
y_true = results_df['y_true'].values

print("\n" + "=" * 40)
print(f"     OFFICIAL PERFORMANCE REPORT ({EVAL_SPLIT})")
print("=" * 40)
print(clip_eval.format_summary(eval_res.summary))
print("\nPrevious run (first-window, argmax, last-epoch weights): accuracy 0.84, "
      "Dangerous P/R 0.88/0.90, Normal recall 0.67\n")

# 2. Per clip score at fixed thresholds -- 0.5 (argmax), 0.67 (ALERT_THRESHOLD),
# 0.90 (SPIKE_THRESHOLD):
#   first-window  = window 0 only (comparable to the old reports)
#   multi max     = highest window (the spike path)
#   multi mean    = average window
#   peak smoothed = highest 4-window rolling average (the sustained path)
scores = {'first-window': results_df['first_p'].values,
          'multi max': results_df['max_p'].values,
          'multi mean': results_df['mean_p'].values,
          'peak smoothed': results_df['peak_smoothed'].values}
ALERT_THRESHOLD = 0.67   # main.py: rolling-average alert
SPIKE_THRESHOLD = 0.90   # main.py: single-window spike
THRESHOLDS = [0.5, ALERT_THRESHOLD, SPIKE_THRESHOLD]
target_names = ['Normal (0)', 'Dangerous (1)']

for name, s in scores.items():
    print(f"##### {name} @ threshold 0.5 #####")
    print(classification_report(y_true, (s >= 0.5).astype(int), labels=[0, 1],
                                target_names=target_names, zero_division=0))

# 3. Confusion matrices: every clip score at every threshold, plus the live
# incident rule itself (the verdict /analyze/video would give).
fig, axes = plt.subplots(len(scores) + 1, len(THRESHOLDS),
                         figsize=(5 * len(THRESHOLDS), 4 * (len(scores) + 1)))
for r, (name, s) in enumerate(scores.items()):
    for c, th in enumerate(THRESHOLDS):
        cm = confusion_matrix(y_true, (s >= th).astype(int), labels=[0, 1])
        print(f"{name} @ {th:.2f}: [[TN={cm[0,0]}, FP={cm[0,1]}], [FN={cm[1,0]}, TP={cm[1,1]}]]  "
              f"Normal recall={cm[0,0]/max(cm[0].sum(),1):.3f}  Dangerous recall={cm[1,1]/max(cm[1].sum(),1):.3f}")
        sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', ax=axes[r, c], cbar=False,
                    xticklabels=['Pred Normal', 'Pred Dangerous'],
                    yticklabels=['Actually Normal', 'Actually Dangerous'])
        axes[r, c].set_title(f"{name} @ {th:.2f}")
    print()
cm_live = confusion_matrix(y_true, results_df['would_alert'].astype(int).values, labels=[0, 1])
sns.heatmap(cm_live, annot=True, fmt='d', cmap='Reds', ax=axes[-1, 0], cbar=False,
            xticklabels=['No alert', 'Alert'], yticklabels=['Actually Normal', 'Actually Dangerous'])
axes[-1, 0].set_title('LIVE incident rule (would_alert)')
for ax in axes[-1, 1:]:
    ax.axis('off')
plt.suptitle(f'SafeSpace System: Confusion Matrices (Danger vs Normal, {EVAL_SPLIT})')
plt.tight_layout()
plt.show()

# 4. Per-source recall: is Weaponized easier or harder than Violence?
print("Dangerous recall by source class (fraction of clips scored >= threshold):")
print(f"{'score':<14} {'thr':>5} {'Violence':>16} {'Weaponized':>16}")
for name, s in scores.items():
    for th in THRESHOLDS:
        row = []
        for cls_name in ['Violence', 'Weaponized']:
            m = (results_df['class'] == cls_name).values
            row.append(f"{(s[m] >= th).mean():.3f} (n={m.sum()})" if m.any() else "n/a")
        print(f"{name:<14} {th:>5.2f} {row[0]:>16} {row[1]:>16}")
print("\nLive incident rule alert rate by class:", eval_res.summary['live_alert_rate_by_class'])

In [ ]:
# --- INSPECT MISSES: what is the model actually failing to see? ---
# Pulls every test clip where the model scored Normal but the true label
# was Dangerous (the ~97 first-window misses from the last eval run), sorted by how
# CONFIDENT the model was in that wrong answer. That distinction matters:
# a miss where the model gave Dangerous only 4% probability is a genuine
# blind spot (never learned to recognize this pattern at all). A miss at
# 47% is a coin-flip the model almost got right -- a fundamentally
# different, less concerning kind of failure, and a candidate for
# threshold tuning rather than a real model weakness.
#
# For each miss, this pulls several frames spanning the FULL raw clip (not
# just the 16 frames the model happened to see this run) so you can check
# a specific hypothesis directly: does the actual dangerous content sit
# somewhere in the clip that jitter could easily have missed, or is this
# genuinely calm-looking footage that's just mislabeled/ambiguous?
import os
import cv2
import numpy as np

MISS_OUTPUT_DIR = "miss_review"
N_CONTEXT_FRAMES = 5  # spread across the whole raw clip, not just what jitter sampled
os.makedirs(MISS_OUTPUT_DIR, exist_ok=True)

def get_context_frames(path, n=N_CONTEXT_FRAMES):
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return []
    positions = [int(total * i / (n - 1)) if n > 1 else total // 2 for i in range(n)]
    positions = [min(p, total - 1) for p in positions]
    frames = []
    for pos in positions:
        cap.set(cv2.CAP_PROP_POS_FRAMES, pos)
        ok, frame = cap.read()
        if ok:
            frames.append((pos, frame))
    cap.release()
    return frames

# Reuses results_df from the evaluation cell -- NO second inference pass.
# MISS_SCORE picks which clip score defines a "miss":
#   'first_p' = first window only (the old definition, ~97 misses last run)
#   'max_p'   = best window anywhere in the clip (mirrors SPIKE)
#   'mean_p'  = average over windows (mirrors the rolling ALERT)
MISS_SCORE = 'max_p'
MISS_THRESHOLD = 0.5

missed = results_df[(results_df['y_true'] == 1) & (results_df[MISS_SCORE] < MISS_THRESHOLD)]
miss_records = [(r.path, r.first_p, r.max_p, getattr(r, MISS_SCORE)) for r in missed.itertuples()]

print(f"\n{len(miss_records)} misses found (true=Dangerous, {MISS_SCORE} < {MISS_THRESHOLD}).")

# Most-confidently-wrong first -- these are the real blind spots to look at.
miss_records.sort(key=lambda r: r[3])

print(f"Saving {len(miss_records)} review strips to {MISS_OUTPUT_DIR}/ ...")
for rank, (path, first_p, max_p, dangerous_prob) in enumerate(miss_records):
    context = get_context_frames(path)
    if not context:
        continue
    thumbs = []
    for pos, frame in context:
        thumb = cv2.resize(frame, (160, 120))
        cv2.putText(thumb, f"f{pos}", (4, 14), cv2.FONT_HERSHEY_SIMPLEX,
                    0.4, (255, 255, 255), 1, cv2.LINE_AA)
        thumbs.append(thumb)
    strip = np.hstack(thumbs)
    header = np.full((24, strip.shape[1], 3), 30, dtype=np.uint8)
    label = f"{os.path.basename(path)}  |  Dangerous prob: first={first_p:.2f} max={max_p:.2f}"
    cv2.putText(header, label, (6, 17), cv2.FONT_HERSHEY_SIMPLEX,
                0.45, (255, 255, 255), 1, cv2.LINE_AA)
    canvas = np.vstack([header, strip])

    out_name = f"{rank:03d}_prob{dangerous_prob:.2f}_{os.path.basename(path)}.jpg"
    cv2.imwrite(os.path.join(MISS_OUTPUT_DIR, out_name), canvas)

print(f"\nDone. Files are named rank_probability_filename.jpg -- sorted so the")
print(f"lowest-numbered files are the model's most confident wrong answers.")
print(f"Start there: those are the clearest signal of what the model has")
print(f"never learned to recognize, versus the higher-probability files near")
print(f"the end of the list, which were closer to a coin flip.")


In [ ]:
from google.colab import files

import json

# 1. Define the filenames for your saved brain
# A new name per run so earlier weights stay around for side-by-side
# comparison with video_calibration.py --model-path. Edit MODEL_STEM freely.
#   - model_filename:   a PLAIN state_dict -- what main.py loads.
#   - sidecar_filename: thresholds + the PREPROCESSING contract. The backend
#     reads it to know how to crop/normalize for this model. Keep it next to
#     the .pth with the same name: without it, the model is treated as v1.
#   - full_filename:    the state_dict plus provenance (epoch, validation
#     report, config flags) for the project record. load_model accepts it
#     too, and reads the contract from its config if no sidecar is present.
# Timestamped + preprocessing-tagged, so it can never overwrite older weights
# (the project folder already has a safespace_bullying_model_v2.pth from an
# earlier model generation -- unrelated to preprocessing "v2").
from datetime import datetime
_preproc_tag = PREPROCESSING.name.replace(',', '_').replace('=', '-')
MODEL_STEM = f"safespace_bullying_model_{datetime.now():%Y%m%d_%H%M}_preproc-{_preproc_tag}"
model_filename = f'{MODEL_STEM}.pth'
sidecar_filename = f'{MODEL_STEM}.json'
full_filename = f'{MODEL_STEM}_full.pth'

# 2. Save the BEST epoch's weights (by validation), not the last.
print(f"Saving best-epoch weights (epoch {best_epoch}) to {model_filename} and {full_filename}...")
torch.save(best_state, model_filename)
torch.save({
    'state_dict': best_state,
    'epoch': best_epoch,
    'val_metrics': best_val_metrics,
    'history': history,
    'config': {
        'seed': SEED,
        'epochs': epochs,
        'USE_CLASS_WEIGHTS': USE_CLASS_WEIGHTS,
        'class_weights': (sqrt_weights.tolist() if USE_CLASS_WEIGHTS else None),
        'UNFREEZE_LAYER4': UNFREEZE_LAYER4,
        'smart_crop': True,
        'preprocessing': preprocessing.sidecar_value(PREPROCESSING),
        'FREEZE_FROZEN_BN': FREEZE_FROZEN_BN,
        'horizontal_flip': True,
        'selection_metric': SELECTION_METRIC,
        'frame_cache': {'format': CACHE_FORMAT, 'fingerprint': cache.fingerprint},
        'val_split': split_approach,
        'num_classes': 2,
        'class_names': ['Normal', 'Dangerous'],
    },
}, full_filename)

# Thresholds start at the defaults; re-run video_calibration.py with the new
# model and tune them here before relying on it.
def _json_safe(o):
    if isinstance(o, float) and not math.isfinite(o):
        return None
    if isinstance(o, dict):
        return {k: _json_safe(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_json_safe(v) for v in o]
    return o

with open(sidecar_filename, 'w') as f:
    json.dump(_json_safe({
        'thresholds': {'alert_threshold': 0.67, 'spike_threshold': 0.90},
        'preprocessing': preprocessing.sidecar_value(PREPROCESSING),
        'best_epoch': best_epoch,
        'selection': {'metric': SELECTION_METRIC, 'value': best_val_value,
                      'validation': best_val_metrics},
        'note': 'Recalibrate thresholds with video_calibration.py before deployment.',
    }), f, indent=2)
print(f"Wrote {sidecar_filename} (preprocessing {PREPROCESSING.name})")

# 3. Download both files from the cloud directly to your local computer
print("Downloading to your computer... check your browser downloads!")
files.download(model_filename)
files.download(sidecar_filename)
files.download(full_filename)

In [11]:
from pathlib import Path
from google.colab import files

folder_path = Path("miss_review")
output_file = Path("miss_review_files.txt")

# Collect only files (skips nested directories)
file_names = [f.name for f in folder_path.iterdir() if f.is_file()]

# Write filenames to text file
with open(output_file, "w", encoding="utf-8") as f:
    f.write("\n".join(file_names))

# Trigger browser download (Google Colab)
files.download(output_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>